# Part 2: Retrieval-Augmented Generation with the Part 1 SLM

This notebook tests whether external information changes the responses of the **same small language model (SLM) selected in Part 1**. It reloads **Architecture / Model #3** from `tiny_transformer.pt`, as trained and selected in `assignment_P1-A7_SLM.ipynb`. RAG supplies retrieved text at inference time; it does not update the model's weights.

The separate `knowledge-base-documents` folder contains five documents about top fiction books by category in 2025. Each document covers one category from [Kirkus Reviews' Best of 2025 lists](https://www.kirkusreviews.com/best-of/2025/):

1. Best Mysteries and Thrillers of 2025
2. Best Book Club Fiction of 2025
3. Best Romance of 2025
4. Best Dystopian Fiction of 2025
5. Best Historical Fiction of 2025

We will ask the SLM five questions about this knowledge base and record its answers **without RAG** as a baseline. Then we will use TF-IDF and cosine similarity to retrieve the Top-1, Top-2, Top-3, and Top-5 documents for the same questions, add the retrieved information to the SLM's prompt, and generate new answers. Finally, we will compare factual accuracy, relevance, and support from the retrieved documents, including how the model's short character-level context window affects the results.

In [1]:
# CELL 1: Load the selected Part 1 SLM from tiny_transformer.pt
from pathlib import Path
import json
import random
import time
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
DEVICE = torch.device("cpu")
TEMPERATURE = 0.7
MAX_NEW_CHARACTERS = 80
TOP_K_VALUES = (1, 2, 3, 5)
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

roots = [Path.cwd(), *Path.cwd().parents]
candidates = roots + [root / "Assignment_P1-A7" / "moveon" / "6-moveon" / "Assignment_P1-A7" for root in roots]
ASSIGNMENT_DIR = next((path for path in candidates if (path / "tiny_transformer.pt").is_file() and (path / "knowledge-base-documents").is_dir()), None)
if ASSIGNMENT_DIR is None:
    raise FileNotFoundError("Cannot locate tiny_transformer.pt beside knowledge-base-documents.")
CHECKPOINT_PATH = ASSIGNMENT_DIR / "tiny_transformer.pt"
KB_DIR = ASSIGNMENT_DIR / "knowledge-base-documents"
CORPUS_PATH = ASSIGNMENT_DIR / "training_corpus.txt"

# Match the architecture used to save the selected Part 1 checkpoint.
class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, feedforward_dim):
        super().__init__()
        self.attention = nn.MultiheadAttention(embedding_dim, num_heads, dropout=0.0, batch_first=True)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.feedforward = nn.Sequential(
            nn.Linear(embedding_dim, feedforward_dim),
            nn.ReLU(),
            nn.Linear(feedforward_dim, embedding_dim)
        )

    def forward(self, x):
        length = x.shape[1]
        mask = torch.triu(torch.ones(length, length, device=x.device, dtype=torch.bool), diagonal=1)
        normalized = self.ln1(x)
        attended, _ = self.attention(normalized, normalized, normalized, attn_mask=mask, need_weights=False)
        x = x + attended
        return x + self.feedforward(self.ln2(x))

class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads, num_layers, feedforward_dim, context_length):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)
        self.blocks = nn.Sequential(*[
            TransformerBlock(embedding_dim, num_heads, feedforward_dim)
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_ids):
        _, length = input_ids.shape
        if length > self.context_length:
            raise ValueError("Input sequence exceeds context length.")
        positions = torch.arange(length, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.blocks(x)
        x = self.ln_final(x)
        return self.output(x)

saved = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
config = saved["config"]
chars = saved["vocab"]
stoi = {char: index for index, char in enumerate(chars)}
itos = {index: char for index, char in enumerate(chars)}
model = TinyTransformerLM(vocab_size=len(chars), **config).to(DEVICE)
model.load_state_dict(saved["model_state"], strict=True)
model.eval()
checkpoint_weights = {name: tensor.detach().clone() for name, tensor in model.state_dict().items()}
CONTEXT_LENGTH = model.context_length
print("Loaded:", CHECKPOINT_PATH)
print("Architecture:", config)
print("Vocabulary:", len(chars), "characters")
print("Context:", CONTEXT_LENGTH, "characters")

Loaded: C:\Users\chant\Desktop\UIUC\INFO 490 - Advanced AI Web App Development\Modules\Week 7\Week 7 Scripts\Week 7 Scripts\Assignment_P1-A7\moveon\6-moveon\Assignment_P1-A7\tiny_transformer.pt
Architecture: {'embedding_dim': 48, 'num_heads': 4, 'num_layers': 1, 'feedforward_dim': 192, 'context_length': 32}
Vocabulary: 27 characters
Context: 32 characters


## Five questions for the baseline and RAG tests

These questions ask for one verifiable fact each. The selected SLM has a **32-character context window**, so the retrieved title and author can fit in the Top-1 prompt. We use the **same wording** without RAG and at every Top-K setting.

1. **Q1 — Mystery and thrillers:** Who wrote *King Of Ashes*?
2. **Q2 — Book-club fiction:** Who wrote *Flashlight*?
3. **Q3 — Dystopian fiction:** Who wrote *The Dream Hotel*?
4. **Q4 — Romance:** Who wrote *Unloved*?
5. **Q5 — Historical fiction:** Who wrote *Isola*?

The correct authors are stored in the separate JSON knowledge base. Baseline prompts contain only these questions; RAG prompts add retrieved facts at inference time.

## Test the SLM Without RAG

The five author questions were sent directly to the saved Part 1 SLM with **no retrieved documents**. Generation used temperature **0.7**, seed **42**, and up to **80 new characters**. The model could initially see only the last **32 characters** of each question.

| Question | SLM Response Without RAG |
|---|---|
| **Q1** — Mystery: Who wrote King Of Ashes? | of iws inework.<br>inference uses previously learned parameters.<br>the university off |
| **Q2** — Book club: Who wrote Flashlight? | and d ducts.<br>small language models can run on local computers.<br>students build dj |
| **Q3** — Dystopian: Who wrote The Dream Hotel? | networks learn from training data.<br>machine learning models contain parameters.<br>t |
| **Q4** — Romance: Who wrote Unloved? | ata.<br>machine learning models contain parameters.<br>the professor teaches artificia |
| **Q5** — Historical: Who wrote Isola? | ind views.<br>training changes the weights of a neural network.<br>inference uses prev |

None of the baseline responses names the requested author. The incomplete endings are the model's actual output at the 80-character limit.

In [2]:
# Baseline: use the loaded Part 1 SLM only. No retrieval or knowledge-base text.
BASELINE_QUESTIONS = [
    ("Q1", "Mystery: Who wrote King Of Ashes?"),
    ("Q2", "Book club: Who wrote Flashlight?"),
    ("Q3", "Dystopian: Who wrote The Dream Hotel?"),
    ("Q4", "Romance: Who wrote Unloved?"),
    ("Q5", "Historical: Who wrote Isola?"),
]

def prepare_slm_prompt(question):
    # The checkpoint vocabulary has only 27 characters; replace unsupported characters with spaces.
    normalized = question.lower()
    fallback = " " if " " in stoi else chars[0]
    normalized = "".join(character if character in stoi else fallback for character in normalized)
    return normalized[-CONTEXT_LENGTH:]

@torch.no_grad()
def generate_baseline(question, max_new_characters=MAX_NEW_CHARACTERS, temperature=TEMPERATURE, seed=SEED):
    if temperature <= 0:
        raise ValueError("Temperature must be positive.")
    visible_prompt = prepare_slm_prompt(question)
    ids = torch.tensor([[stoi[character] for character in visible_prompt]], dtype=torch.long, device=DEVICE)
    model.eval()
    torch.manual_seed(seed)
    generated = []
    for _ in range(max_new_characters):
        logits = model(ids[:, -CONTEXT_LENGTH:])[:, -1, :] / temperature
        next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        ids = torch.cat((ids, next_id), dim=1)
        generated.append(itos[next_id.item()])
    return "".join(generated).strip(), visible_prompt

baseline_rows = []
for question_id, question in BASELINE_QUESTIONS:
    response, visible_prompt = generate_baseline(question)
    baseline_rows.append({"Question": question_id, "Question Text": question,
                          "SLM Response Without RAG": response, "Visible Prompt": visible_prompt})

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df[["Question", "Question Text", "SLM Response Without RAG"]])
print("All five responses used the saved Part 1 SLM without retrieved documents.")
print("Visible prompt endings:", baseline_df[["Question", "Visible Prompt"]].to_dict("records"))

,Question,Question Text,SLM Response Without RAG
0,Q1,Mystery: Who wrote King Of Ashes?,of iws inework.\ninference uses previously lea...
1,Q2,Book club: Who wrote Flashlight?,and d ducts.\nsmall language models can run on...
2,Q3,Dystopian: Who wrote The Dream Hotel?,networks learn from training data.\nmachine le...
3,Q4,Romance: Who wrote Unloved?,ata.\nmachine learning models contain paramete...
4,Q5,Historical: Who wrote Isola?,ind views.\ntraining changes the weights of a ...


All five responses used the saved Part 1 SLM without retrieved documents.
Visible prompt endings: [{'Question': 'Q1', 'Visible Prompt': 'ystery  who wrote king of ashes '}, {'Question': 'Q2', 'Visible Prompt': 'book club  who wrote flashlight '}, {'Question': 'Q3', 'Visible Prompt': 'pian  who wrote the dream hotel '}, {'Question': 'Q4', 'Visible Prompt': 'romance  who wrote unloved '}, {'Question': 'Q5', 'Visible Prompt': 'historical  who wrote isola '}]


## Applying RAG to the SLM

The code below loads the **five separate JSON documents** and indexes their **45 individual book records** with TF-IDF. Cosine similarity retrieves Top-1, Top-2, Top-3, and Top-5 books for each of the **same five questions** used in the baseline. It packs retrieved title/author facts into the SLM's **32-character** input window. The saved model generates each response with **no weight updates**.

In the table, **↵ represents a newline** in the exact prompt visible to the model at the start of generation. Higher K leaves fewer characters per book. The source document and the selected fact for every retrieved book are also recorded in the code output.

| Question | Top-K | Retrieved Books | Visible Prompt | Generated Response | Response Quality (1–10) |
|---|---:|---|---|---|---:|
| Q1 | 1 | 01_02 King Of Ashes | `king of ashes s.a. cosby ↵ by` | the inference intelligence can generate text.<br>the model predicts characters usin | 1 |
| Q1 | 2 | 01_02 King Of Ashes; 01_08 Buried Above Ground | `mike ripley s.a. cosby ↵ by` | the inelps models use attention mechanisms.<br>neural networks learn from training | 1 |
| Q1 | 3 | 01_02 King Of Ashes; 01_08 Buried Above Ground; 01_05 Detective Aunty | `u ma jala mike ripl s.a. cosb ↵ by` | the stems use attention mechanisms.<br>neural networks learn from training data.<br>ma | 1 |
| Q1 | 5 | 01_02 King Of Ashes; 01_08 Buried Above Ground; 01_05 Detective Aunty; 04_10 Ladies In Hating; 03_05 What We Can Know | `ian m alexa u ma  mike  s.a.  ↵ by` | the university offers courses in informatics.<br>students experiment with language | 1 |
| Q2 | 1 | 02_01 Flashlight | `flashlight susan choi ↵ by` | the s learn python programming.<br>students develop artificial intelligence systems | 1 |
| Q2 | 2 | 02_01 Flashlight; 02_08 Gaysians | `mike curato susan choi ↵ by` | the stems use attention mechanisms.<br>neural networks learn from training data.<br>ma | 1 |
| Q2 | 3 | 02_01 Flashlight; 02_08 Gaysians; 02_07 Moderation | `elaine ca mike cura susan cho ↵ by` | the s artificial intelligence.<br>students create innovative startup applications. | 1 |
| Q2 | 5 | 02_01 Flashlight; 02_08 Gaysians; 02_07 Moderation; 02_05 Fresh, Green Life; 03_05 What We Can Know | `ian m sebas elain mike  susan ↵ by` | the university offers courses in informatics.<br>students experiment with language | 1 |
| Q3 | 1 | 03_03 The Dream Hotel | `the dream hotel laila lalami ↵ by` | the university offers courses in informatics.<br>students experiment with language | 1 |
| Q3 | 2 | 03_03 The Dream Hotel; 05_08 The Listeners | `maggie stiefva laila lalami ↵ by` | the university offers courses in informatics.<br>students experiment with language | 1 |
| Q3 | 3 | 03_03 The Dream Hotel; 05_08 The Listeners; 03_02 Brother BrontÃ« | `fernando  maggie st laila lal ↵ by` | the inference uses  previously learned parameters.<br>the university offers courses | 1 |
| Q3 | 5 | 03_03 The Dream Hotel; 05_08 The Listeners; 03_02 Brother BrontÃ«; 03_01 The Book Of Guilt; 03_07 Sacrament | `susan cathe ferna maggi laila ↵ by` | the university offers courses in informatics.<br>students experiment with language | 1 |
| Q4 | 1 | 04_01 Unloved | `unloved peyton corinne ↵ by` | the inference intelligence can generate text.<br>the model predicts characters usin | 1 |
| Q4 | 2 | 04_01 Unloved; 04_02 Chaos | `constance fay peyton corinne ↵ by` | the stems echaracters using probabilities.<br>attention helps models use contextual | 1 |
| Q4 | 3 | 04_01 Unloved; 04_02 Chaos; 04_09 Can'T Get Enough | `kennedy r constance peyton co ↵ by` | the s artificial intelligence.<br>students create innovative startup applications. | 1 |
| Q4 | 5 | 04_01 Unloved; 04_02 Chaos; 04_09 Can'T Get Enough; 04_05 It Had To Be Him; 04_04 A Tropical Rebel Gets The Duke | `adria adib  kenne const peyto ↵ by` | the inference intelligence can generate text.<br>the model predicts characters usin | 1 |
| Q5 | 1 | 05_01 Isola | `isola allegra goodman ↵ by` | the university offers courses in informatics.<br>students experiment with language | 1 |
| Q5 | 2 | 05_01 Isola; 05_07 The Antidote | `karen russell allegra goodma ↵ by` | the university offers courses in informatics.<br>students experiment with language | 1 |
| Q5 | 3 | 05_01 Isola; 05_07 The Antidote; 05_04 The Buffalo Hunter Hunter | `stephen g karen rus allegra g ↵ by` | the stems use attention mechanisms.<br>neural networks learn from training data.<br>ma | 1 |
| Q5 | 5 | 05_01 Isola; 05_07 The Antidote; 05_04 The Buffalo Hunter Hunter; 05_05 Fonseca; 05_08 The Listeners | `maggi jessi steph karen alleg ↵ by` | the stems use attention mechanisms.<br>neural networks learn from training data.<br>ma | 1 |

**Quality assessment:** The intended book ranked first for each question, and its full title and author fit in every Top-1 prompt. None of the 20 generated responses names the requested author. Each therefore receives **1/10** for factual accuracy and relevance. As generation continues, the initial retrieved fact also falls out of the model's 32-character window.

In [3]:
# Book-level RAG: index each book inside the five separate JSON source documents.
# TF-IDF and cosine similarity follow the class retrieval pipeline.
book_records = []
source_documents = []
seen_book_ids = set()
for path in sorted(KB_DIR.glob("*.json")):
    with path.open("r", encoding="utf-8") as handle:
        source = json.load(handle)
    source_id = str(source["document_id"])
    source_title = str(source["document_title"])
    source_documents.append({"id": source_id, "title": source_title, "file": path.name})
    for book in source["books"]:
        book_id = str(book["book_id"])
        if book_id in seen_book_ids:
            raise ValueError(f"Duplicate book ID: {book_id}")
        seen_book_ids.add(book_id)
        title = str(book["book_title"])
        author = str(book["author"])
        searchable = " ".join([
            title, title, source_title, author,
            str(book.get("genre", "")),
            str(book.get("synopsis", "")),
            str(book.get("review_synthesis", ""))
        ])
        book_records.append({"book_id": book_id, "source_document": source_id,
                             "title": title, "author": author,
                             "search_text": searchable})
if len(source_documents) != 5 or len(book_records) < 5:
    raise ValueError("Expected five category files containing book records.")
corpus_text = CORPUS_PATH.read_text(encoding="utf-8").lower()
assert all(book["search_text"].lower() not in corpus_text for book in book_records)

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
book_vectors = vectorizer.fit_transform([book["search_text"] for book in book_records])

def retrieve_books(question, k):
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, book_vectors).ravel()
    ranked = sorted(range(len(book_records)), key=lambda i: (-scores[i], book_records[i]["book_id"]))
    return [(book_records[i], float(scores[i])) for i in ranked[:k]]

def slm_supported(text):
    fallback = " " if " " in stoi else chars[0]
    return "".join(char if char in stoi else fallback for char in text.lower())

def compact_book_prompt(retrieved):
    # Preserve a short factual title/author pair when it fits. At higher K,
    # prioritize author characters and record exactly what the model receives.
    suffix = "\nby"
    budget = CONTEXT_LENGTH - len(suffix)
    separators = len(retrieved) - 1
    base, remainder = divmod(budget - separators, len(retrieved))
    fragments = []
    visible_evidence = []
    for rank, (book, _) in enumerate(retrieved):
        share = base + (rank < remainder)
        title_author = slm_supported(book["title"] + " " + book["author"])
        author = slm_supported(book["author"])
        fragment = title_author if len(title_author) <= share else author[:share]
        fragments.append(fragment)
        visible_evidence.append({"book_id": book["book_id"],
                                 "source_document": book["source_document"],
                                 "title": book["title"],
                                 "author": book["author"],
                                 "visible_text": fragment})
    # Rank 1 is nearest the final cue.
    prompt = " ".join(reversed(fragments)) + suffix
    assert len(prompt) <= CONTEXT_LENGTH
    return prompt, visible_evidence

rag_rows = []
for question_id, question in BASELINE_QUESTIONS:
    for k in TOP_K_VALUES:
        retrieved = retrieve_books(question, k)
        prompt, visible_evidence = compact_book_prompt(retrieved)
        response, visible_prompt = generate_baseline(prompt)
        rag_rows.append({
            "Question": question_id,
            "Top-K": k,
            "Retrieved Book IDs": [book["book_id"] for book, _ in retrieved],
            "Retrieved Titles": [book["title"] for book, _ in retrieved],
            "Source Documents": [book["source_document"] for book, _ in retrieved],
            "Similarity Scores": [round(score, 3) for _, score in retrieved],
            "Visible Evidence": visible_evidence,
            "Visible Prompt": visible_prompt,
            "Generated Response": response,
        })
rag_df = pd.DataFrame(rag_rows)
assert len(rag_df) == 20
assert all(torch.equal(checkpoint_weights[name], tensor)
           for name, tensor in model.state_dict().items())
print("Indexed", len(book_records), "books from", len(source_documents), "JSON documents.")
print("Generated 20 responses with the unchanged Part 1 SLM.")
display(rag_df[["Question", "Top-K", "Retrieved Book IDs", "Retrieved Titles",
                "Visible Evidence", "Generated Response"]])

Indexed 45 books from 5 JSON documents.
Generated 20 responses with the unchanged Part 1 SLM.


,Question,Top-K,Retrieved Book IDs,Retrieved Titles,Visible Evidence,Generated Response
0,Q1,1,[01_02],[King Of Ashes],"[{'book_id': '01_02', 'source_document': '01',...",the inference intelligence can generate text.\...
1,Q1,2,"[01_02, 01_08]","[King Of Ashes, Buried Above Ground]","[{'book_id': '01_02', 'source_document': '01',...",the inelps models use attention mechanisms.\nn...
2,Q1,3,"[01_02, 01_08, 01_05]","[King Of Ashes, Buried Above Ground, Detective...","[{'book_id': '01_02', 'source_document': '01',...",the stems use attention mechanisms.\nneural ne...
3,Q1,5,"[01_02, 01_08, 01_05, 04_10, 03_05]","[King Of Ashes, Buried Above Ground, Detective...","[{'book_id': '01_02', 'source_document': '01',...",the university offers courses in informatics.\...
4,Q2,1,[02_01],[Flashlight],"[{'book_id': '02_01', 'source_document': '02',...",the s learn python programming.\nstudents deve...
5,Q2,2,"[02_01, 02_08]","[Flashlight, Gaysians]","[{'book_id': '02_01', 'source_document': '02',...",the stems use attention mechanisms.\nneural ne...
6,Q2,3,"[02_01, 02_08, 02_07]","[Flashlight, Gaysians, Moderation]","[{'book_id': '02_01', 'source_document': '02',...",the s artificial intelligence.\nstudents creat...
7,Q2,5,"[02_01, 02_08, 02_07, 02_05, 03_05]","[Flashlight, Gaysians, Moderation, Fresh, Gree...","[{'book_id': '02_01', 'source_document': '02',...",the university offers courses in informatics.\...
8,Q3,1,[03_03],[The Dream Hotel],"[{'book_id': '03_03', 'source_document': '03',...",the university offers courses in informatics.\...
9,Q3,2,"[03_03, 05_08]","[The Dream Hotel, The Listeners]","[{'book_id': '03_03', 'source_document': '03',...",the university offers courses in informatics.\...


## Analysis

1. **Did providing retrieved information change the SLM's responses?** Yes. All 20 RAG continuations differed from the corresponding no-RAG baseline responses, showing that the added prompt text affected generation.

2. **Did RAG improve factual accuracy?** No. Although Top-1 retrieved the intended book and its author for all five questions, none of the 20 generated responses named the requested author; every RAG response was rated 1/10 for this task.

3. **Did retrieving more documents always improve the responses?** No. Top-2, Top-3, and Top-5 did not produce a correct author, and adding books reduced the number of prompt characters available for each book's fact.

4. **How did the SLM's context-length limitation affect the results?** Its 32-character window could hold a full title and author in each Top-1 prompt, but facts from multiple books had to be shortened at higher K. As generation continued, the retrieved fact also moved out of the window.

5. **Why might RAG behave differently with a small character-level model compared with a larger pretrained language model?** This model learned next-character patterns from a tiny, repetitive corpus rather than learning to answer questions from supplied evidence, and its vocabulary and 32-character window constrain what it can read. A larger pretrained model may be better able to interpret and use retrieved passages, though retrieval alone does not guarantee a correct answer.